# One-model orientation-specific runner

Train exactly one of the 10 hierarchy models. Orientation-specific action/position models can use CNN, TCN, InceptionTime-style CNN, or MiniRocket.

In [ ]:
from __future__ import annotations

import os
import json
import joblib
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.metrics import f1_score, classification_report, confusion_matrix

try:
    !pip install sktime[all_extras] -q
    import sktime
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real

except Exception as exc:
    print("BayesSearchCV unavailable. Use search_mode='grid'.", exc)

import sys
sys.path.append('/kaggle/input/datasets/keithmarange/onemodelstuff/')
sys.path.append('/kaggle/input/cmi-competition-code')

import data_utils
import utils_one_model_orientation_specific as utils

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
# ============================================================
# Config
# ============================================================

search_mode = "bayesian"       # "grid" or "bayesian"
run_mode = "search"        # "search", "best_params", or "model"
selected_model_name = "action_lie_back"

# For action/position orientation-specific models only:
# "cnn", "tcn", "inception", "minirocket"
selected_model_type = "minirocket"

random_state = 42
holdout_size = 0.2
use_train_subset = True
train_sequence_frac = 0.4
n_cv_splits = 2
n_iter_bayes = 10
n_jobs = 1
verbose = 3

pipe_name = "sequence_builder"
classifier_name = "classifier"
corrector_name = "orientation_corrector"

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# Use when run_mode == "best_params". Accepts either:
# 1) dict keyed by selected_model_name, or
# 2) a raw params dict.
previous_best_params_path = None

# Use when run_mode == "model"
previous_model_path = None

save_fitted_model = True
save_cv_results = True

model_catalog = {
    "target": {"target": "is_target", "data": "all", "orientation": None},
    "orientation": {"target": "orientation", "data": "target", "orientation": None},

    "action_lie_back": {"target": "gesture_action", "data": "target", "orientation": "Lie on Back"},
    "action_lie_side_non_dom": {"target": "gesture_action", "data": "target", "orientation": "Lie on Side - Non Dominant"},
    "action_seated_lean_face_down": {"target": "gesture_action", "data": "target", "orientation": "Seated Lean Non Dom - FACE DOWN"},
    "action_seated_straight": {"target": "gesture_action", "data": "target", "orientation": "Seated Straight"},

    "position_lie_back": {"target": "gesture_position", "data": "target", "orientation": "Lie on Back"},
    "position_lie_side_non_dom": {"target": "gesture_position", "data": "target", "orientation": "Lie on Side - Non Dominant"},
    "position_seated_lean_face_down": {"target": "gesture_position", "data": "target", "orientation": "Seated Lean Non Dom - FACE DOWN"},
    "position_seated_straight": {"target": "gesture_position", "data": "target", "orientation": "Seated Straight"},
}

if selected_model_name not in model_catalog:
    raise ValueError(f"selected_model_name must be one of: {list(model_catalog)}")

if selected_model_name in ["target", "orientation"] and selected_model_type != "cnn":
    print("Note: target/orientation usually use cnn. Continuing with selected_model_type anyway.")

print("timestamp:", timestamp)
print("selected_model_name:", selected_model_name)
print("selected_model_type:", selected_model_type)
print("run_mode:", run_mode)

In [ ]:
# ============================================================
# Load data
# ============================================================

data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
raw_test_df = pd.read_csv(data_root / "test.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
test_demo_df = pd.read_csv(data_root / "test_demographics.csv")

print("raw_train_df:", raw_train_df.shape)
print("train_demo_df:", train_demo_df.shape)
print("train sequences:", raw_train_df["sequence_id"].nunique())
print("subjects:", raw_train_df["subject"].nunique())

In [ ]:
# ============================================================
# Base dataframe + helper targets
# ============================================================

train_df = raw_train_df.set_index("row_id").copy(deep=True)

train_df.loc[:, "gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df.loc[:, "gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df.loc[:, "is_target"] = train_df["sequence_type"].eq("Target").astype(int)

train_df.loc[:, "hierarchical_gesture"] = np.where(
    train_df["sequence_type"].eq("Target"),
    train_df["gesture"],
    "Non-Target",
)

print("sequence_type counts")
print(train_df.drop_duplicates("sequence_id")["sequence_type"].value_counts())

print("\ngesture_action classes")
print(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture_action"].value_counts())

print("\ngesture_position classes")
print(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture_position"].value_counts())

print("\norientation classes")
print(train_df.loc[train_df["sequence_type"].eq("Target"), "orientation"].value_counts())

In [ ]:
# ============================================================
# Subject holdout split + optional train subset
# ============================================================

seq_meta = (
    train_df
    .drop_duplicates("sequence_id")
    [["sequence_id", "subject", "sequence_type", "gesture", "hierarchical_gesture", "is_target"]]
    .reset_index(drop=True)
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=holdout_size,
    random_state=random_state,
)

train_seq_idx, holdout_seq_idx = next(
    splitter.split(
        seq_meta,
        y=seq_meta["hierarchical_gesture"],
        groups=seq_meta["subject"],
    )
)

train_seq_ids = seq_meta.loc[train_seq_idx, "sequence_id"]
holdout_seq_ids = seq_meta.loc[holdout_seq_idx, "sequence_id"]

train_model_df = train_df.loc[train_df["sequence_id"].isin(train_seq_ids)].copy()
holdout_df = train_df.loc[train_df["sequence_id"].isin(holdout_seq_ids)].copy()

if use_train_subset:
    train_seq_meta = (
        train_model_df
        .drop_duplicates("sequence_id")
        [["sequence_id", "subject", "sequence_type", "gesture", "is_target"]]
        .reset_index(drop=True)
    )

    sampled_train_seq_ids = (
        train_seq_meta
        .groupby("sequence_type", group_keys=False)
        .sample(frac=train_sequence_frac, random_state=random_state)["sequence_id"]
    )

    train_model_df = train_model_df.loc[
        train_model_df["sequence_id"].isin(sampled_train_seq_ids)
    ].copy()


target_only_train_df = train_model_df.loc[train_model_df["sequence_type"].eq("Target")].copy()
target_only_holdout_df = holdout_df.loc[holdout_df["sequence_type"].eq("Target")].copy()

print("full sequences:", train_df["sequence_id"].nunique())
print("train sequences:", train_model_df["sequence_id"].nunique())
print("holdout sequences:", holdout_df["sequence_id"].nunique())
print("target-only train sequences:", target_only_train_df["sequence_id"].nunique())
print("target-only holdout sequences:", target_only_holdout_df["sequence_id"].nunique())
print("train subjects:", train_model_df["subject"].nunique())
print("holdout subjects:", holdout_df["subject"].nunique())
print("subject overlap:", len(set(train_model_df["subject"]) & set(holdout_df["subject"])))

In [ ]:
# ============================================================
# Select the single model dataset
# ============================================================

model_info = model_catalog[selected_model_name]
target_name = model_info["target"]
orientation_filter = model_info["orientation"]

if model_info["data"] == "all":
    selected_train_df = train_model_df.copy()
    selected_holdout_df = holdout_df.copy()
else:
    selected_train_df = target_only_train_df.copy()
    selected_holdout_df = target_only_holdout_df.copy()

if orientation_filter is not None:
    selected_train_df = selected_train_df.loc[
        selected_train_df["orientation"].eq(orientation_filter)
    ].copy()
    selected_holdout_df = selected_holdout_df.loc[
        selected_holdout_df["orientation"].eq(orientation_filter)
    ].copy()

selected_train_seq = selected_train_df.drop_duplicates("sequence_id").copy()
selected_holdout_seq = selected_holdout_df.drop_duplicates("sequence_id").copy()

print("target_name:", target_name)
print("orientation_filter:", orientation_filter)
print("train sequences:", selected_train_df["sequence_id"].nunique())
print("holdout sequences:", selected_holdout_df["sequence_id"].nunique())
print("train class counts:")
print(selected_train_seq[target_name].value_counts(dropna=False))
print("holdout class counts:")
print(selected_holdout_seq[target_name].value_counts(dropna=False))

if selected_train_df["sequence_id"].nunique() < n_cv_splits:
    raise ValueError("Not enough sequences for the requested CV splits.")

In [ ]:
# ============================================================
# Param spaces
# ============================================================

if search_mode == "bayesian":
    assert BayesSearchCV is not None, "BayesSearchCV unavailable. Use search_mode='grid'."

    sequence_param_space = {
        f"{pipe_name}__acc_modes": Categorical([
            "smoothed|velocity|displacement|jerk",
        ]),
        f"{pipe_name}__rotation_modes": Categorical([
            "quaternion|euler|rot6d|angular_velocity",
        ]),
        f"{pipe_name}__sampling_rate": Integer(5, 120),
        f"{pipe_name}__interp_mode": Categorical(["ffill"]),
        f"{pipe_name}__standardize": Categorical(["mean_std"]),
        f"{pipe_name}__linear_acc_mode": Categorical(["baseline"]),
        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([True]),
        f"{pipe_name}__tof_mode": Categorical(["pooled_stats"]),
        f"{pipe_name}__tof_fill_mode": Categorical(["far_255"]),
        f"{pipe_name}__thm_mode": Categorical(["centered"]),
        f"{pipe_name}__motion_filter_mode": Categorical(["extended_kalman"]),
        f"{pipe_name}__kalman_process_noise": Real(1e-5, 1e-1, prior="log-uniform"),
        f"{pipe_name}__kalman_measurement_noise": Real(1e-4, 2.0, prior="log-uniform"),
        f"{pipe_name}__use_dead_reckoning": Categorical([True]),
        f"{pipe_name}__clip_value": Categorical([50.0, 100.0]),
        f"{pipe_name}__window_size": Integer(2, 20),
        f"{pipe_name}__smooth_alpha": Categorical([0.8]),
    }

    cnn_param_space = {
        **sequence_param_space,
        f"{classifier_name}__maxlen": Integer(5, 260),
        f"{classifier_name}__conv_filters": Categorical(["512-512-512", "256-512-512", "256-512-512-512"]),
        f"{classifier_name}__kernel_sizes": Categorical(["5-5-5", "5-5-5-5", "7-5-5-3"]),
        f"{classifier_name}__pool_sizes": Categorical(["none", "2", "2-2"]),
        f"{classifier_name}__use_batch_norm": Categorical([True]),
        f"{classifier_name}__spatial_dropout": Categorical([0.2, 0.25]),
        f"{classifier_name}__dense_units": Categorical(["64", "128", "128-64"]),
        f"{classifier_name}__dropout": Categorical([0.2, 0.45]),
        f"{classifier_name}__learning_rate": Categorical([3e-5, 5e-5, 8e-5]),
        f"{classifier_name}__batch_size": Categorical([32]),
        f"{classifier_name}__epochs": Categorical([150]),
        f"{classifier_name}__patience": Categorical([8]),
        f"{classifier_name}__use_mixup": Categorical([False]),
    }

    tcn_param_space = {
        **sequence_param_space,
        f"{classifier_name}__maxlen": Integer(120, 260),
        f"{classifier_name}__tcn_filters": Categorical([128, 192, 256]),
        f"{classifier_name}__kernel_size": Categorical([3, 5, 7]),
        f"{classifier_name}__dilations": Categorical(["1-2-4-8", "1-2-4-8-16"]),
        f"{classifier_name}__tcn_blocks": Categorical([1, 2]),
        f"{classifier_name}__dense_units": Categorical(["64", "128", "128-64"]),
        f"{classifier_name}__dropout": Categorical([0.2, 0.45]),
        f"{classifier_name}__spatial_dropout": Categorical([0.1, 0.2]),
        f"{classifier_name}__learning_rate": Categorical([3e-5, 5e-5, 8e-5]),
        f"{classifier_name}__batch_size": Categorical([32]),
        f"{classifier_name}__epochs": Categorical([150]),
        f"{classifier_name}__patience": Categorical([8]),
        f"{classifier_name}__use_mixup": Categorical([False]),
    }

    inception_param_space = {
        **sequence_param_space,
        f"{classifier_name}__maxlen": Integer(120, 260),
        f"{classifier_name}__n_filters": Categorical([32, 64, 96]),
        f"{classifier_name}__kernel_sizes": Categorical(["9-19-39", "7-15-31", "5-11-23"]),
        f"{classifier_name}__inception_blocks": Categorical([1, 2, 3]),
        f"{classifier_name}__bottleneck_size": Categorical([32, 64]),
        f"{classifier_name}__dense_units": Categorical(["64", "128", "128-64"]),
        f"{classifier_name}__dropout": Categorical([0.2, 0.45]),
        f"{classifier_name}__spatial_dropout": Categorical([0.1, 0.2]),
        f"{classifier_name}__learning_rate": Categorical([3e-5, 5e-5, 8e-5]),
        f"{classifier_name}__batch_size": Categorical([32]),
        f"{classifier_name}__epochs": Categorical([150]),
        f"{classifier_name}__patience": Categorical([8]),
        f"{classifier_name}__use_mixup": Categorical([False]),
    }

    minirocket_param_space = {
        **sequence_param_space,
        f"{classifier_name}__maxlen": Integer(5, 220),
        f"{classifier_name}__num_kernels": Integer(84, 20000),
    }

elif search_mode == "grid":
    sequence_param_space = {
        f"{pipe_name}__acc_modes": ["smoothed|velocity|displacement|jerk"],
        f"{pipe_name}__rotation_modes": ["quaternion|euler|rot6d|angular_velocity"],
        f"{pipe_name}__sampling_rate": [97],
        f"{pipe_name}__interp_mode": ["linear"],
        f"{pipe_name}__standardize": ["mean_std"],
        f"{pipe_name}__linear_acc_mode": ["baseline"],
        f"{pipe_name}__use_acc_magnitude": [True],
        f"{pipe_name}__use_linear_acc_magnitude": [True],
        f"{pipe_name}__tof_mode": ["pooled_stats"],
        f"{pipe_name}__tof_fill_mode": ["far_255"],
        f"{pipe_name}__thm_mode": ["centered"],
        f"{pipe_name}__motion_filter_mode": ["extended_kalman"],
        f"{pipe_name}__kalman_process_noise": [0.001],
        f"{pipe_name}__kalman_measurement_noise": [0.1],
        f"{pipe_name}__use_dead_reckoning": [True],
        f"{pipe_name}__clip_value": [50.0],
        f"{pipe_name}__window_size": [2],
        f"{pipe_name}__smooth_alpha": [0.8],
    }

    cnn_param_space = {
        **sequence_param_space,
        f"{classifier_name}__maxlen": [220],
        f"{classifier_name}__conv_filters": ["256-512-512-512"],
        f"{classifier_name}__kernel_sizes": ["5-5-5"],
        f"{classifier_name}__pool_sizes": ["none"],
        f"{classifier_name}__use_batch_norm": [True],
        f"{classifier_name}__spatial_dropout": [0.2],
        f"{classifier_name}__dense_units": ["64"],
        f"{classifier_name}__dropout": [0.2],
        f"{classifier_name}__learning_rate": [5e-5],
        f"{classifier_name}__batch_size": [32],
        f"{classifier_name}__epochs": [80],
        f"{classifier_name}__patience": [8],
        f"{classifier_name}__use_mixup": [False],
    }

    tcn_param_space = {
        **sequence_param_space,
        f"{classifier_name}__maxlen": [220],
        f"{classifier_name}__tcn_filters": [192],
        f"{classifier_name}__kernel_size": [5],
        f"{classifier_name}__dilations": ["1-2-4-8"],
        f"{classifier_name}__tcn_blocks": [1],
        f"{classifier_name}__dense_units": ["64"],
        f"{classifier_name}__dropout": [0.2],
        f"{classifier_name}__spatial_dropout": [0.2],
        f"{classifier_name}__learning_rate": [5e-5],
        f"{classifier_name}__batch_size": [32],
        f"{classifier_name}__epochs": [80],
        f"{classifier_name}__patience": [8],
        f"{classifier_name}__use_mixup": [False],
    }

    inception_param_space = {
        **sequence_param_space,
        f"{classifier_name}__maxlen": [220],
        f"{classifier_name}__n_filters": [64],
        f"{classifier_name}__kernel_sizes": ["9-19-39"],
        f"{classifier_name}__inception_blocks": [2],
        f"{classifier_name}__bottleneck_size": [32],
        f"{classifier_name}__dense_units": ["64"],
        f"{classifier_name}__dropout": [0.2],
        f"{classifier_name}__spatial_dropout": [0.2],
        f"{classifier_name}__learning_rate": [5e-5],
        f"{classifier_name}__batch_size": [32],
        f"{classifier_name}__epochs": [80],
        f"{classifier_name}__patience": [8],
        f"{classifier_name}__use_mixup": [False],
    }

    minirocket_param_space = {
        **sequence_param_space,
        f"{classifier_name}__maxlen": [160, 220],
        f"{classifier_name}__num_kernels": [10000],
    }

else:
    raise ValueError(f"Unknown search_mode: {search_mode}")

if selected_model_type == "cnn":
    param_space = cnn_param_space
elif selected_model_type == "tcn":
    param_space = tcn_param_space
elif selected_model_type == "inception":
    param_space = inception_param_space
elif selected_model_type == "minirocket":
    param_space = minirocket_param_space
else:
    raise ValueError("selected_model_type must be 'cnn', 'tcn', 'inception', or 'minirocket'")

print("param count:", len(param_space))
print("selected param space keys:")
for k in param_space:
    print(k)

In [ ]:
# ============================================================
# Build selected estimator
# ============================================================

if selected_model_type == "cnn":
    classifier = utils.KerasAugmentedCNN1DSequenceClassifier(
        target=target_name,
        verbose=verbose,
        random_state=random_state,
    )
elif selected_model_type == "tcn":
    classifier = utils.KerasTCNSequenceClassifier(
        target=target_name,
        verbose=verbose,
        random_state=random_state,
    )
elif selected_model_type == "inception":
    classifier = utils.KerasInceptionTimeSequenceClassifier(
        target=target_name,
        verbose=verbose,
        random_state=random_state,
    )
elif selected_model_type == "minirocket":
    classifier = utils.MiniRocketSequenceClassifier(
        target=target_name,
        random_state=random_state,
    )
else:
    raise ValueError("Unknown selected_model_type")

pipeline = Pipeline([
    (corrector_name, utils.SensorOrientationCorrector(demo_df=train_demo_df)),
    (pipe_name, utils.AdvancedMultiDomainSequenceExtractor()),
    (classifier_name, classifier),
])

pipeline

In [ ]:
# ============================================================
# Optional previous params/model loading
# ============================================================

previous_best_params = None

if previous_best_params_path is not None:
    with open(previous_best_params_path, "r") as f:
        params_blob = json.load(f)

    if selected_model_name in params_blob:
        previous_best_params = params_blob[selected_model_name]
    else:
        previous_best_params = params_blob

    print("loaded previous params from:", previous_best_params_path)
    print("params keys:", len(previous_best_params))

if run_mode == "model":
    if previous_model_path is None:
        raise ValueError("previous_model_path must be set when run_mode == 'model'.")
    loaded_estimator = joblib.load(previous_model_path)
    search = SimpleNamespace(
        best_estimator_=loaded_estimator,
        best_params_=getattr(loaded_estimator, "get_params", lambda: {})(),
        best_score_=np.nan,
        cv_results_={"params": ["loaded_model"], "mean_test_score": [np.nan]},
    )
    results_df = pd.DataFrame(search.cv_results_)
    print("loaded fitted model from:", previous_model_path)

elif run_mode == "best_params":
    if previous_best_params is None:
        raise ValueError("previous_best_params_path must be supplied when run_mode == 'best_params'.")
    pipeline.set_params(**previous_best_params)
    y_train = selected_train_df[["sequence_id", target_name]].copy()
    pipeline.fit(selected_train_df, y_train)
    refit_score = pipeline.score(selected_train_df, y_train)
    search = SimpleNamespace(
        best_estimator_=pipeline,
        best_params_=previous_best_params,
        best_score_=refit_score,
        cv_results_={"params": [previous_best_params], "mean_test_score": [refit_score]},
    )
    results_df = pd.DataFrame(search.cv_results_)
    print("loaded previous best params and refit selected model")
    print("train refit score:", refit_score)

elif run_mode == "search":
    print("search will run in the next cell")

else:
    raise ValueError("run_mode must be 'search', 'best_params', or 'model'.")

In [ ]:
# ============================================================
# Run search for selected model only
# ============================================================

if run_mode == "search":
    cv = GroupKFold(n_splits=n_cv_splits)
    y_train = selected_train_df[["sequence_id", target_name]].copy()
    groups = selected_train_df["subject"].to_numpy()

    if search_mode == "bayesian":
        search = BayesSearchCV(
            estimator=pipeline,
            search_spaces=param_space,
            n_iter=n_iter_bayes,
            scoring=None,
            cv=cv,
            n_jobs=n_jobs,
            refit=True,
            random_state=random_state,
            verbose=verbose,
            error_score=np.nan,
            return_train_score=True,
        )
    elif search_mode == "grid":
        search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_space,
            scoring=None,
            cv=cv,
            n_jobs=n_jobs,
            refit=True,
            verbose=verbose,
            error_score=np.nan,
            return_train_score=True,
        )
    else:
        raise ValueError(f"Unknown search_mode: {search_mode}")

    search.fit(selected_train_df, y_train, groups=groups)
    results_df = pd.DataFrame(search.cv_results_)

print("best score:", search.best_score_)
print("best params:")
for k, v in search.best_params_.items():
    print(k, ":", v)

In [ ]:
# ============================================================
# Save CV results and fitted model
# ============================================================

safe_model_type = selected_model_type.replace(" ", "_")
results_path = results_dir / f"cv_results_{selected_model_name}_{safe_model_type}_{timestamp}.csv"
model_path = results_dir / f"fitted_{selected_model_name}_{safe_model_type}_{timestamp}.joblib"
params_path = results_dir / f"best_params_{selected_model_name}_{safe_model_type}_{timestamp}.json"

if save_cv_results:
    results_df.to_csv(results_path, index=False)
    print("saved cv results:", results_path)

if save_fitted_model and run_mode != "model":
    joblib.dump(search.best_estimator_, model_path)
    print("saved fitted model:", model_path)

with open(params_path, "w") as f:
    json.dump(search.best_params_, f, indent=2, default=str)
print("saved best params:", params_path)

In [ ]:
# ============================================================
# Holdout evaluation for selected model
# ============================================================

if selected_holdout_df is not None and not selected_holdout_df.empty:
    holdout_seq = (
        selected_holdout_df
        .drop_duplicates("sequence_id")
        [["sequence_id", "subject", "sequence_type", "gesture", "gesture_action", "gesture_position", "orientation", target_name]]
        .reset_index(drop=True)
    )

    preds = search.best_estimator_.predict(selected_holdout_df)
    holdout_seq.loc[:, "prediction"] = preds.astype(str)

    holdout_macro_f1 = f1_score(
        holdout_seq[target_name],
        holdout_seq["prediction"],
        average="macro",
    )

    print("holdout macro F1:", round(holdout_macro_f1, 4))
    print("\nclassification report")
    print(classification_report(holdout_seq[target_name], holdout_seq["prediction"]))

    pred_path = results_dir / f"holdout_predictions_{selected_model_name}_{safe_model_type}_{timestamp}.csv"
    holdout_seq.to_csv(pred_path, index=False)
    print("saved holdout predictions:", pred_path)
else:
    holdout_macro_f1 = np.nan
    print("No holdout rows for this selected model. Skipping holdout evaluation.")

In [ ]:
# ============================================================
# Compact summary
# ============================================================

summary_df = pd.DataFrame([
    {
        "timestamp": timestamp,
        "selected_model_name": selected_model_name,
        "selected_model_type": selected_model_type,
        "search_mode": search_mode,
        "run_mode": run_mode,
        "target_name": target_name,
        "orientation_filter": orientation_filter,
        "cv_best_score": search.best_score_,
        "holdout_macro_f1": holdout_macro_f1,
        "train_sequences": selected_train_df["sequence_id"].nunique(),
        "holdout_sequences": selected_holdout_df["sequence_id"].nunique(),
        "train_subjects": selected_train_df["subject"].nunique(),
        "holdout_subjects": selected_holdout_df["subject"].nunique(),
        "best_params": json.dumps(search.best_params_, default=str),
    }
])

summary_path = results_dir / f"summary_{selected_model_name}_{safe_model_type}_{timestamp}.csv"
summary_df.to_csv(summary_path, index=False)
display(summary_df)
print("saved summary:", summary_path)